In [4]:
from typing import Any, List
from dotenv import load_dotenv
import json
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper

## Building Agent Class


In [ ]:
class Agent:
    def __init__(
        self,
        role: str = "You are a helpful assistant.",
        instructions: str = "You are a helpful assistant.",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[tool] = None,
    ):
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools if tools is not None else []

        load_dotenv()

        self.llm = LLM(
            model=self.model,
            temperature=self.temperature,
            tools=self.tools,
        )

    def invoke(self, user_message: str) -> str:
        messages = [
            SystemMessage(
                content=(
                    f"You are an AI agent with the role: {self.role}. "
                    f"Your instructions are: {self.instructions}."
                )
            )
        ]

        messages.append(UserMessage(content=user_message))

        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        while ai_message.tool_calls:
            for call in ai_message.tool_calls:
                function_name = call.function_name
                function_args = json.loads(call.function.arguments)
                tool_call_id = call.id

                tool = next((t for t in self.tools if t.name == function_name), None)
                if tool:
                    result = tool(**function_args)
                    messages.append(
                        ToolMessage(
                            content=json.dumps(result),
                            tool_call_id=tool_call_id,
                            name=function_name,
                        )
                    )

            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)

        for m in messages:
            print(m)

        return ai_message.content

## Testing Your Agent

Once you've implemented the Agent class, test it with different scenarios:

In [5]:
agent = Agent(role="Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

AttributeError: 'Agent' object has no attribute 'invoke'